In [2]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import nltk

## 1. Loading the dataset

In [3]:
#Loading the dataset
file_path = "Datasets/amazon_review_small.txt"
df = pd.read_csv(file_path, sep=",", quotechar='"', engine="python", header=None, names=["rating","title", "review"])



In [4]:
print(df.head(2))
print(f"\nLoaded {len(df)} rows and {df.shape[1]} columns from {file_path}")

   rating                    title  \
0       1          mens ultrasheer   
1       4  Surprisingly delightful   

                                              review  
0  This model may be ok for sedentary types, but ...  
1  This is a fast read filled with unexpected hum...  

Loaded 650000 rows and 3 columns from Datasets/amazon_review_small.txt


## 2. Exploring Data

In [5]:
df["rating"].value_counts()

rating
1    130000
4    130000
2    130000
3    130000
5    130000
Name: count, dtype: int64

As we cam see we have rating 1-5 and each rating has equal number of reviews. 

In [6]:
df.isnull().sum()

rating     0
title     26
review     0
dtype: int64

In [7]:
#Fill missing values with empty string
df["title"] = df["title"].fillna("")

## 3. Cleaning Data

In [8]:
#Creating a copy of dataset and Combining title and review into a single text column
df_work = df.copy()
df_work["text"] = df_work["title"] + " " + df_work["review"]
#Selecting only the text and rating columns for further processing
df_work = df_work[["text", "rating"]]
df_work["text"][0]

"mens ultrasheer This model may be ok for sedentary types, but I'm active and get around alot in my job - consistently found these stockings rolled up down by my ankles! Not Good!! Solution: go with the standard compression stocking, 20-30, stock #114622. Excellent support, stays up and gives me what I need. Both pair of these also tore as I struggled to pull them up all the time. Good riddance/bad investment!"

In [9]:
#Function to clean the text data
import re
def clean_text(text):
    # Remove HTML tags
    text = re.sub(r"<.*?>", "", text)
    # Remove non-alphanumeric characters with spaces to preserve proper word separation
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    # Convert to lowercase
    text = text.lower()
    #Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()
    return text


In [10]:
#Apply cleaning function to the text column
df_work["text"] = df_work["text"].apply(clean_text)

## 4. Splitting Data

In [13]:
# Separate features (X) and target (y)
X = df_work["text"]
y = df_work["rating"]

# First split: 80% train, 20% test
from sklearn.model_selection import train_test_split
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# Second split (on training data): 80% train, 20% validation
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.20, random_state=42
)

print("Train size:", X_train.shape[0])
print("Validation size:", X_val.shape[0])
print("Test size:", X_test.shape[0])

Train size: 416000
Validation size: 104000
Test size: 130000


## 5. Vectorization
### Strategy

Initially, we will use TF-IDF for the **base model** and check accuracy. Later, if needed, we will switch to **Word2Vec** or other advanced embedding techniques.
### What is TF-IDF?

**TF-IDF (Term Frequency – Inverse Document Frequency)** converts raw text into numeric vectors that machine learning models can process.

It scores each word in a document using two components:

- **TF (Term Frequency):** How often a word appears in a single document (review).  
  $TF(t, d) = \dfrac{\text{count of } t \text{ in } d}{\text{total words in } d}$

- **IDF (Inverse Document Frequency):** Penalizes words that appear in many documents (e.g. "the", "is"), giving higher weight to rare, meaningful words.  
  $IDF(t) = \log\left(\dfrac{1 + N}{1 + df(t)}\right) + 1$

- **TF-IDF score:**  
  $\text{TF-IDF}(t, d) = TF(t, d) \times IDF(t)$

### How `TfidfVectorizer` works

1. Scans all training reviews and builds a **vocabulary** of unique words.
2. Ranks words by frequency and keeps the top **`max_features`** words.
3. Represents each review as a vector of TF-IDF scores — one value per vocabulary word.
4. Words not seen during training are ignored at transform time.

### Settings used here

| Parameter | Value | Meaning |
|---|---|---|
| `max_features` | 5000 | Keep only top 5000 most frequent words as vocab |
| `analyzer` | word *(default)* | Tokenize by words, not characters |
| `token_pattern` | `\b[a-zA-Z0-9]{2,}\b` *(default)* | Words with 2+ alphanumeric chars |


In [ ]:
# Vectorization using TF-IDF (fit on train, transform train/validation/test)
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000)  # Limit to top 5000 features. This is the number of unique words that will be considered in the model. 
X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)
X_test_tfidf = vectorizer.transform(X_test)

print("X_train_tfidf shape:", X_train_tfidf.shape)
print("X_val_tfidf shape:", X_val_tfidf.shape)
print("X_test_tfidf shape:", X_test_tfidf.shape)

X_train_tfidf shape: (455000, 5000)
X_test_tfidf shape: (195000, 5000)


In [ ]:
# Displaying the word to index mapping for the TF-IDF vectorizer
word_to_index = vectorizer.vocabulary_
print("Sample word to index mapping:", list(word_to_index.items())[:10])


Sample word to index mapping: [('pre', np.int64(3346)), ('dance', np.int64(1102)), ('teacher', np.int64(4373)), ('and', np.int64(234)), ('never', np.int64(2949)), ('get', np.int64(1877)), ('tired', np.int64(4506)), ('of', np.int64(3022)), ('this', np.int64(4458)), ('cd', np.int64(725))]


## 6. Using a base model to train data.